In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df = spark.read.format("delta").load('abfss://878eac13-677f-422e-b9fa-bae10de1d354@onelake.dfs.fabric.microsoft.com/62582fb9-8cbf-4f28-aa83-e002f40371ec/Tables/raw/RAW_SALES_DTL')
df.show()

StatementMeta(, 81c282c4-a479-4090-969e-7e80dd86e266, 4, Finished, Available, Finished, False)

+--------+------------+------------+----------+------------+--------+------------+
|Trxn_ID#|  !Date_Ref!|PROD_CODE_ID|Cust_ID_99|Store_Loc_ID|Qty_Sold|_Unit_Price_|
+--------+------------+------------+----------+------------+--------+------------+
|   T1001|  15-01-2026|      P_9921|     C-551|        S_01|       2|        1200|
|   T1002|  16-01-2026|      P_1022|      NULL|        S_02|       1|       450.5|
|   T1003|Jan 17, 2026|      P_9921|     C-882|        S_01|       1|        1200|
|   T1004|  2026.01.18|      P_4451|     C-551|        S_01|       5|         200|
|   T1005|   19-Jan-26|      P_1022|     C-102|        S_02|       1|     INR 450|
|   T1006|  20-01-2026|      P_9921|     C-551|        S_03|       1|        1200|
|   T1007|    20260121|      P_1022|      NULL|        S_01|       2|         450|
|   T1008|  01-22-2026|      P_4451|     C-999|        S_03|      10|    $ 180.00|
|   T1009|  23-01-2026|      P_8810|     C-882|        S_02|       1|       55000|
|   

In [6]:
df = df.selectExpr("`Trxn_ID#` as tran_id","`!Date_Ref!` as date_ref","PROD_CODE_ID as prod_code_id",
"Cust_ID_99 as cust_id","Store_Loc_ID as store_loc_id","Qty_Sold as qty_sold","_Unit_Price_ as unit_price")
df.show()

StatementMeta(, 81c282c4-a479-4090-969e-7e80dd86e266, 8, Finished, Available, Finished, False)

+-------+------------+------------+-------+------------+--------+----------+
|tran_id|    date_ref|prod_code_id|cust_id|store_loc_id|qty_sold|unit_price|
+-------+------------+------------+-------+------------+--------+----------+
|  T1001|  15-01-2026|      P_9921|  C-551|        S_01|       2|      1200|
|  T1002|  16-01-2026|      P_1022|   NULL|        S_02|       1|     450.5|
|  T1003|Jan 17, 2026|      P_9921|  C-882|        S_01|       1|      1200|
|  T1004|  2026.01.18|      P_4451|  C-551|        S_01|       5|       200|
|  T1005|   19-Jan-26|      P_1022|  C-102|        S_02|       1|   INR 450|
|  T1006|  20-01-2026|      P_9921|  C-551|        S_03|       1|      1200|
|  T1007|    20260121|      P_1022|   NULL|        S_01|       2|       450|
|  T1008|  01-22-2026|      P_4451|  C-999|        S_03|      10|  $ 180.00|
|  T1009|  23-01-2026|      P_8810|  C-882|        S_02|       1|     55000|
|  T1010|  24-01-2026|      P_9921|  C-551|        S_01|       1|      1100|

In [11]:
from pyspark.sql.functions import to_date, coalesce, col, date_format
df = df.withColumn(
   "date_ref",
   date_format(
       coalesce(
           to_date(col("date_ref"), "MMM dd, yyyy"),
           to_date(col("date_ref"), "yyyy.MM.dd"), 
           to_date(col("date_ref"), "dd-MM-yyyy"),
           to_date(col("date_ref"),"dd-MMM-yy"),
           to_date(col("date_ref"),"yyyyMMdd"),
           to_date(col("date_ref"),"MM-dd-yyyy")

       ),
       "yyyy-MM-dd"
   )
)
df.show()

StatementMeta(, 81c282c4-a479-4090-969e-7e80dd86e266, 13, Finished, Available, Finished, False)

+-------+----------+------------+-------+------------+--------+----------+
|tran_id|  date_ref|prod_code_id|cust_id|store_loc_id|qty_sold|unit_price|
+-------+----------+------------+-------+------------+--------+----------+
|  T1001|2026-01-15|      P_9921|  C-551|        S_01|       2|      1200|
|  T1002|2026-01-16|      P_1022|   NULL|        S_02|       1|     450.5|
|  T1003|2026-01-17|      P_9921|  C-882|        S_01|       1|      1200|
|  T1004|2026-01-18|      P_4451|  C-551|        S_01|       5|       200|
|  T1005|2026-01-19|      P_1022|  C-102|        S_02|       1|   INR 450|
|  T1006|2026-01-20|      P_9921|  C-551|        S_03|       1|      1200|
|  T1007|2026-01-21|      P_1022|   NULL|        S_01|       2|       450|
|  T1008|2026-01-22|      P_4451|  C-999|        S_03|      10|  $ 180.00|
|  T1009|2026-01-23|      P_8810|  C-882|        S_02|       1|     55000|
|  T1010|2026-01-24|      P_9921|  C-551|        S_01|       1|      1100|
|  T1011|2026-01-25|     

In [21]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "cust_id",
    when(col("cust_id") == "NULL", "Unknown").otherwise(col("cust_id"))
)
df.show()

StatementMeta(, 81c282c4-a479-4090-969e-7e80dd86e266, 23, Finished, Available, Finished, False)

+-------+----------+------------+-------+------------+--------+----------+
|tran_id|  date_ref|prod_code_id|cust_id|store_loc_id|qty_sold|unit_price|
+-------+----------+------------+-------+------------+--------+----------+
|  T1001|2026-01-15|      P_9921|  C-551|        S_01|       2|      1200|
|  T1002|2026-01-16|      P_1022|Unknown|        S_02|       1|     450.5|
|  T1003|2026-01-17|      P_9921|  C-882|        S_01|       1|      1200|
|  T1004|2026-01-18|      P_4451|  C-551|        S_01|       5|       200|
|  T1005|2026-01-19|      P_1022|  C-102|        S_02|       1|   INR 450|
|  T1006|2026-01-20|      P_9921|  C-551|        S_03|       1|      1200|
|  T1007|2026-01-21|      P_1022|Unknown|        S_01|       2|       450|
|  T1008|2026-01-22|      P_4451|  C-999|        S_03|      10|  $ 180.00|
|  T1009|2026-01-23|      P_8810|  C-882|        S_02|       1|     55000|
|  T1010|2026-01-24|      P_9921|  C-551|        S_01|       1|      1100|
|  T1011|2026-01-25|     

In [30]:
from pyspark.sql.functions import regexp_replace, col

df = df.withColumn(
    "unit_price",
    regexp_replace(col("unit_price"), r"[^0-9.]", "")
)

df = df.withColumn("qty_sold",df["qty_sold"].cast("integer"))
df = df.withColumn("unit_price",df["unit_price"].cast("double"))
df.show()
df.dtypes


StatementMeta(, 81c282c4-a479-4090-969e-7e80dd86e266, 32, Finished, Available, Finished, False)

+-------+----------+------------+-------+------------+--------+----------+
|tran_id|  date_ref|prod_code_id|cust_id|store_loc_id|qty_sold|unit_price|
+-------+----------+------------+-------+------------+--------+----------+
|  T1001|2026-01-15|      P_9921|  C-551|        S_01|       2|    1200.0|
|  T1002|2026-01-16|      P_1022|Unknown|        S_02|       1|     450.5|
|  T1003|2026-01-17|      P_9921|  C-882|        S_01|       1|    1200.0|
|  T1004|2026-01-18|      P_4451|  C-551|        S_01|       5|     200.0|
|  T1005|2026-01-19|      P_1022|  C-102|        S_02|       1|     450.0|
|  T1006|2026-01-20|      P_9921|  C-551|        S_03|       1|    1200.0|
|  T1007|2026-01-21|      P_1022|Unknown|        S_01|       2|     450.0|
|  T1008|2026-01-22|      P_4451|  C-999|        S_03|      10|     180.0|
|  T1009|2026-01-23|      P_8810|  C-882|        S_02|       1|   55000.0|
|  T1010|2026-01-24|      P_9921|  C-551|        S_01|       1|    1100.0|
|  T1011|2026-01-25|     

[('tran_id', 'string'),
 ('date_ref', 'string'),
 ('prod_code_id', 'string'),
 ('cust_id', 'string'),
 ('store_loc_id', 'string'),
 ('qty_sold', 'int'),
 ('unit_price', 'double')]